# SkinAI — DenseNet121 분류 모델 학습

**Google Colab / 로컬 환경 모두 지원**
- Colab: Drive 마운트 → 레포 클론 → 심링크
- 로컬: 현재 디렉토리 그대로 사용

In [1]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    # Colab 런타임 임시 경로 (세션 종료 시 소실)
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    # 로컬: 노트북이 위치한 레포 루트 그대로 사용
    LOCAL_ROOT = str(Path.cwd())
    PROJECT_ROOT = LOCAL_ROOT

print(f"환경       : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경       : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [2]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Drive 마운트 경로 (Colab 런타임 전용)
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    if not Path(COLAB_ROOT).exists():
        !git clone https://github.com/kyoe-23/skin_ai.git {COLAB_ROOT}
    else:
        # 이미 존재하면 최신 코드로 업데이트
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 4.33 KiB | 2.17 MiB/s, done.
From https://github.com/kyoe-23/skin_ai
   fe0d25c..b0fca47  main       -> origin/main
Updating fe0d25c..b0fca47
Fast-forward
 ai/training/classifier/config.py                  |   2 +-
 skinai_docs/data-dataset-15-memoized-starlight.md | 220 ++++++++++++++++++++
 skinai_docs/dataset_classification_strategy.md    | 224 --------------------
 skinai_docs/dual_model_analysis.md                | 236 ----------------------
 4 files changed, 221 insertions(+), 461 deletions(-)
 create mode 100644 skinai_docs/data-dataset-15-memoized-starlight.md
 delete mode 100644 skinai_docs/dataset_classification_strategy.md
 delete mode 100644 skinai_docs/dual_model_analysis.md


In [7]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 경로 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    # Colab 런타임: Drive에 있는 dataset_14를 심링크로 연결
    os.makedirs("data", exist_ok=True)
    DRIVE_DATASET = f"{DRIVE_ROOT}/data/dataset_14"
    if Path(DRIVE_DATASET).exists():
        !ln -sfn {DRIVE_DATASET} data/dataset_14
        print("data/dataset_14 심링크 생성 완료")
    else:
        print(f"경고: Drive 데이터셋 없음 — {DRIVE_DATASET}")
else:
    # 로컬: data/dataset_14 존재 여부만 확인
    if not Path("data/dataset_14").exists():
        print("경고: data/dataset_14/ 없음 — AI Hub 데이터를 먼저 배치하세요")
    else:
        print("data/dataset_14/ 확인 완료")

!ls data/

현재 디렉토리: /content/colab_skin_ai
data/dataset_14 심링크 생성 완료
dataset_14  processed


In [8]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

# Grad-CAM (추론 서버 사용 시)
# !pip install -q pytorch-grad-cam

In [ ]:
# ── 셀 6-1: 학습 실행 ─────────────────────────────────────────────
# CSV의 zip_path(로컬 절대경로)를 --root_dir로 실행 환경 경로에 맞게 재매핑

# DenseNet121 (기본)
!python -m ai.training.classifier.train \
    --backbone densenet121 \
    --num_epochs 100 \
    --batch_size 256 \
    --root_dir {PROJECT_ROOT}

# EfficientNet-B3 (비교)
# !python -m ai.training.classifier.train \
#     --backbone efficientnet_b3 \
#     --num_epochs 100 \
#     --batch_size 256 \
#     --root_dir {PROJECT_ROOT}

# 세션 만료 후 재개
# !python -m ai.training.classifier.train \
#    --backbone densenet121 \
#    --num_epochs 100 \
#    --batch_size 256 \
#    --resume ai/checkpoints/aihub/epoch_N.pth \
#    --root_dir {PROJECT_ROOT}


INFO [INFO] CUDA 사용
AI Hub 08-14 분류 모델 학습
  backbone : densenet121
  device   : cuda
  epochs   : 100
  batch    : 256
  lr       : 0.001
  warmup   : 3 epochs
  scheduler: CosineAnnealingLR (T_max=97)
INFO Dataset 로드: data/processed/train.csv (4800건, direction=front)
INFO Dataset 로드: data/processed/val.csv (600건, direction=front)

  Train: 4800건
  Val  : 600건
INFO   Model     : DenseNet
INFO   Total     : 6,960,006 params
INFO   Trainable : 6,960,006 params

학습 시작...

[Epoch 01/100] Train Loss: 0.899 | Train Top-1: 65.7%
                    Val   Loss: 2.443 | Val   Top-1: 42.5% | Val Top-3: 92.0%
                    시간: 82.8s | LR: 0.001000
INFO   체크포인트 저장: ai/checkpoints/aihub/best.pth
                    Best 모델 저장 (Top-1: 42.54%)

[Epoch 02/100] Train Loss: 0.490 | Train Top-1: 82.6%
                    Val   Loss: 0.756 | Val   Top-1: 71.1% | Val Top-3: 97.2%
                    시간: 80.0s | LR: 0.001000
INFO   체크포인트 저장: ai/checkpoints/aihub/best.pth
                    Best 모델 저장

In [25]:
# ── 셀 6-2 ─────────────────────────────────────────────
import json
from pathlib import Path

ckpt_dir = Path("ai/checkpoints/aihub")

# 주기 체크포인트 목록
epoch_ckpts = sorted(ckpt_dir.glob("epoch_*.pth"))
print("주기 체크포인트:")
for p in epoch_ckpts:
    print(f"  {p.name}")

# training_log.json에서 실제 마지막 에폭 확인
log_path = ckpt_dir / "training_log.json"
if log_path.exists():
    with open(log_path) as f:
        log = json.load(f)
    last = log["history"][-1]
    print(f"\n마지막 에폭: {last['epoch']}  |  val_top1: {last['val_top1']*100:.2f}%")
    print(f"best 에폭:   {log['best_epoch']}  |  best_top1: {log['best_val_top1']*100:.2f}%")

주기 체크포인트:
  epoch_10.pth
  epoch_15.pth
  epoch_5.pth

마지막 에폭: 19  |  val_top1: 75.92%
best 에폭:   9  |  best_top1: 79.89%


In [20]:
# ── 셀 7: 평가 + Threshold 최적화 ──────────────────────────────
!python -m ai.testing.evaluate \
    --checkpoint ai/checkpoints/aihub/best.pth \
    --root_dir {PROJECT_ROOT}

!python -m ai.testing.threshold_opt \
    --checkpoint ai/checkpoints/aihub/best.pth \
    --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/val.csv (600건, direction=front)
평가 데이터: val.csv (600건)

 SkinAI 분류 모델 평가 결과 (AI Hub 08-14)
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 83.17%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 98.17%
 Macro F1-Score : 0.8308
 Macro AUC      : 0.9692
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.8391   0.7300   0.7807   0.9611
 아토피피부염         0.8469   0.8300   0.8384   0.9744
 여드름            0.7344   0.9400   0.8246   0.9781
 주사             0.9333   0.8400   0.8842   0.9828
 지루피부염          0.6842   0.6500   0.6667   0.9186
 정상             0.9804   1.0000   0.9901   1.0000
/content/colab_skin_ai/ai/testing/evaluate.py:115: UserWarning: Glyph 44148 (\N{HANGUL SYLLABLE GEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/colab_skin_ai/ai/test

In [ ]:
# ── 셀 8: 체크포인트 Drive 저장 + 로컬 다운로드 (Colab 전용) ────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from google.colab import files
    from pathlib import Path

    CKPT_SRC = f"{COLAB_ROOT}/ai/results"
    CKPT_DST = f"{DRIVE_ROOT}/ai/results"

    # 1단계: Drive 백업
    shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
    print(f"Drive 저장 완료: {CKPT_DST}")

    # 2단계: 로컬 다운로드
    ckpt_dir = Path(CKPT_SRC)
    download_targets = (
        sorted(ckpt_dir.glob("*.pth"))
        + sorted(ckpt_dir.glob("*.json"))
        + sorted(ckpt_dir.glob("*.png"))
    )

    print(f"\n로컬 다운로드 시작 ({len(download_targets)}개)...")
    for f in download_targets:
        print(f"  다운로드: {f.name}")
        files.download(str(f))

    print("다운로드 완료 — 로컬 ai/results/ 에 배치하세요")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/ 에 저장됨")

체크포인트 저장 완료: /content/drive/MyDrive/skin_ai/ai/checkpoints/aihub
